# Système de Recommandation TripAdvisor — Baseline BM25

**Architecture modulaire** : chaque section est encapsulée pour brancher facilement un second modèle NLP.

## Pipeline
1. Chargement & Nettoyage des données
2. Stratégie de normalisation du nombre d'avis par lieu
3. Split Train / Test (50/50 par lieu)
4. Construction des représentations textuelles (indexation)
5. Baseline BM25
6. Évaluation — Niveau 1 (typeR) & Niveau 2 (catégories fines)
7. Interface pour brancher un modèle alternatif

---
## 0. Installation des dépendances

In [2]:
%pip install rank-bm25 nltk langdetect tqdm --quiet scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 1. Imports & Configuration globale

In [ ]:
import pandas as pd
import numpy as np
import re
import ast
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Callable

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from tqdm import tqdm
import os
from pathlib import Path

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ─── Configuration ───────────────────────────────────────────────────────────
RANDOM_SEED       = 42
# absolute data path (hardcoded since search failed)
data_dir = Path(r"c:\Users\axell\OneDrive\Documents\Sup\S2\IR&NLP\IR-NPL_Project1\TripAdvisorTrainingDataProject1")
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory not found at {data_dir}")

REVIEWS_FILE = str(data_dir / 'reviews83325.csv')
PLACES_FILE  = str(data_dir / 'Tripadvisor.csv')

LANG_FILTER       = 'en'          # Garder uniquement les avis en anglais
MAX_REVIEWS_PLACE = 50            # Stratégie cap : max avis par lieu
TOP_TFIDF_WORDS   = 100           # Stratégie TF-IDF : top mots par lieu
STRATEGY          = 'cap'         # 'cap' | 'tfidf'  ← changez ici

np.random.seed(RANDOM_SEED)
print('Configuration chargée')
print('Using review file:', REVIEWS_FILE)
print('Using places file :', PLACES_FILE)

✅ Configuration chargée
Using review file: c:\Users\axell\OneDrive\Documents\Sup\S2\IR&NLP\IR-NPL_Project1\TripAdvisorTrainingDataProject1\reviews83325.csv
Using places file : c:\Users\axell\OneDrive\Documents\Sup\S2\IR&NLP\IR-NPL_Project1\TripAdvisorTrainingDataProject1\Tripadvisor.csv


---
## 2. Chargement & Nettoyage des données

In [4]:
# ─── 2.1 Chargement ──────────────────────────────────────────────────────────
reviews_raw = pd.read_csv(REVIEWS_FILE, low_memory=False)
places_raw  = pd.read_csv(PLACES_FILE,  low_memory=False)

print(f'Reviews  : {reviews_raw.shape[0]:,} lignes  | colonnes : {list(reviews_raw.columns)}')
print(f'Places   : {places_raw.shape[0]:,} lignes  | colonnes : {list(places_raw.columns)}')

Reviews  : 340,385 lignes  | colonnes : ['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']
Places   : 3,761 lignes  | colonnes : ['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbSc

In [5]:
# ─── 2.2 Filtrage langue & nettoyage reviews ─────────────────────────────────
reviews = reviews_raw.copy()

# Filtre langue
reviews = reviews[reviews['langue'].str.lower() == LANG_FILTER].copy()
print(f'Avis EN uniquement : {len(reviews):,}')

# gestion colonne texte manquante
if 'text' not in reviews.columns:
    # essayer de détecter un nom alternatif fréquemment utilisé
    candidates = [c for c in reviews.columns if c.lower().startswith('text') or c.lower().startswith('review')]
    if candidates:
        print(f"Renommage automatique de la colonne '{candidates[0]}' en 'text'")
        reviews = reviews.rename(columns={candidates[0]: 'text'})
    else:
        raise KeyError(f"Colonne 'text' introuvable dans reviews, colonnes disponibles : {reviews.columns.tolist()}")

# Supprimer les lignes sans texte
reviews = reviews.dropna(subset=['text']).copy()
reviews['text'] = reviews['text'].astype(str).str.strip()
reviews = reviews[reviews['text'].str.len() > 10].copy()

# Normalisation de la clé de jointure
reviews['idplace'] = reviews['idplace'].astype(str).str.strip()

print(f'Après nettoyage : {len(reviews):,} avis')

Avis EN uniquement : 153,071
Renommage automatique de la colonne 'review' en 'text'
Après nettoyage : 153,066 avis


In [6]:
# ─── 2.3 Nettoyage places & normalisation clé ────────────────────────────────
places = places_raw.copy()
places['id'] = places['id'].astype(str).str.strip()

# Parser les colonnes liste (stockées comme string '[1,2,3]')
def safe_parse_list(val):
    """Convertit une colonne liste sérialisée en liste Python."""
    if pd.isna(val) or val == '' or val == 'nan':
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [str(val)]

LIST_COLS = ['activiteSubCategorie', 'activiteSubType', 'restaurantType', 'cuisine']
for col in LIST_COLS:
    if col in places.columns:
        places[col] = places[col].apply(safe_parse_list)

print(f'Places nettoyées : {len(places):,}')
places[['id', 'typeR'] + [c for c in LIST_COLS if c in places.columns]].head(10)

Places nettoyées : 3,761


,id,typeR,activiteSubCategorie,activiteSubType,restaurantType
0,188467,A,[47],[163],[]
1,188468,A,[47],[163],[]
2,188470,A,"[(26, 47, 51)]","[(34, 144)]",[]
3,188471,A,[26],[137],[]
4,188472,A,[47],[10],[]
5,188493,A,[26],[144],[]
6,188679,A,[47],"[(3, 175, 17)]",[]
7,188738,H,[],[],[]
8,188745,H,[],[],[]
9,188758,A,[49],[161],[]


In [7]:
# ─── 2.4 Jointure ────────────────────────────────────────────────────────────
# On joint uniquement sur les places qui ont au moins un avis
# Construire dynamiquement la liste de colonnes à fusionner en ne prenant
# que celles présentes dans le dataframe `places` (certaines sont optionnelles).
required_cols = ['id', 'typeR']
optional_cols = ['activiteSubCategorie', 'activiteSubType',
                 'restaurantType', 'cuisine', 'priceRange']
meta_cols = required_cols + [c for c in optional_cols if c in places.columns]

reviews_with_meta = reviews.merge(
    places[meta_cols],
    left_on='idplace', right_on='id',
    how='inner'
)

# Lieux valides = ceux ayant au moins un avis EN
valid_place_ids = reviews_with_meta['idplace'].unique()
places = places[places['id'].isin(valid_place_ids)].copy().reset_index(drop=True)

print(f'Avis jointure : {len(reviews_with_meta):,}')
print(f'Lieux valides : {len(places):,}')
print('Distribution typeR :', reviews_with_meta['typeR'].value_counts().to_dict())

Avis jointure : 153,066
Lieux valides : 1,835
Distribution typeR : {'A': 70059, 'R': 44507, 'AP': 24779, 'H': 13721}


---
## 3. Stratégie de normalisation du nombre d'avis

In [8]:
# ─── 3.1 Visualisation du déséquilibre ───────────────────────────────────────
review_counts = reviews_with_meta.groupby('idplace').size()
print('Stats avis/lieu :')
print(review_counts.describe().to_string())
print(f'\nLieux avec >100 avis : {(review_counts > 100).sum()}')
print(f'Lieux avec <5  avis  : {(review_counts < 5).sum()}')

Stats avis/lieu :
count     1835.000000
mean        83.414714
std        828.874642
min          1.000000
25%          2.000000
50%          8.000000
75%         36.000000
max      33878.000000

Lieux avec >100 avis : 244
Lieux avec <5  avis  : 763


In [9]:
# ─── 3.2 Stratégie CAP — garder MAX_REVIEWS_PLACE avis par lieu ──────────────
def strategy_cap(df: pd.DataFrame, max_reviews: int, seed: int = 42) -> pd.DataFrame:
    """
    Pour chaque lieu, échantillonne aléatoirement jusqu'à `max_reviews` avis.
    Avantage  : simple, préserve la diversité des avis.
    Limite    : perd de l'information pour les lieux très populaires.
    """
    return (
        df.groupby('idplace', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max_reviews), random_state=seed))
          .reset_index(drop=True)
    )


# ─── 3.3 Stratégie TF-IDF — top N mots représentatifs ────────────────────────
def strategy_tfidf_keywords(df: pd.DataFrame, top_n: int = 100) -> pd.DataFrame:
    """
    Concatène tous les avis d'un lieu, puis extrait les `top_n` mots
    TF-IDF les plus représentatifs pour créer un pseudo-document.
    Avantage  : représentation dense et discriminante.
    Limite    : perte du signal de fréquence et du contexte.
    """
    place_docs = df.groupby('idplace')['text'].apply(lambda t: ' '.join(t)).reset_index()
    place_docs.columns = ['idplace', 'full_text']

    vectorizer = TfidfVectorizer(max_features=None, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(place_docs['full_text'])
    feature_names = np.array(vectorizer.get_feature_names_out())

    keyword_docs = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray().flatten()
        top_idx = row.argsort()[::-1][:top_n]
        keywords = ' '.join(feature_names[top_idx])
        keyword_docs.append({'idplace': place_docs.iloc[i]['idplace'], 'text': keywords})

    return pd.DataFrame(keyword_docs)


# ─── 3.4 Application de la stratégie choisie ─────────────────────────────────
if STRATEGY == 'cap':
    reviews_balanced = strategy_cap(reviews_with_meta, MAX_REVIEWS_PLACE, RANDOM_SEED)
    # Concaténer les avis par lieu pour former un document
    place_documents = (
        reviews_balanced
        .groupby('idplace')['text']
        .apply(lambda t: ' '.join(t.astype(str)))
        .reset_index()
        .rename(columns={'text': 'document'})
    )
elif STRATEGY == 'tfidf':
    tfidf_df = strategy_tfidf_keywords(reviews_with_meta, TOP_TFIDF_WORDS)
    place_documents = tfidf_df.rename(columns={'text': 'document'})
else:
    raise ValueError(f'Stratégie inconnue : {STRATEGY}')

# Joindre les métadonnées
place_documents = place_documents.merge(places, left_on='idplace', right_on='id', how='inner')
print(f'Documents construits : {len(place_documents):,} lieux')
place_documents[['idplace', 'typeR', 'document']].head(10)

Documents construits : 1,835 lieux


C:\Users\axell\AppData\Local\Temp\ipykernel_3408\4088005306.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max_reviews), random_state=seed))


,idplace,typeR,document
0,10041740,R,We loved our meal here and would return in a h...
1,10055668,R,"Nice staff, wonderful advices, and incredible ..."
2,10089740,A,My family and another have been planning a tri...
3,10105969,A,This is a small art gallery on Place des Vosge...
4,10131114,R,This oas a cute place for teatime. The cakes ...
5,1013459,R,"The creamy ice-creams (chocolate, chocolate no..."
6,10163512,A,My heart beats a little faster when I walk acr...
7,10163562,A,We crossed this bridge when walking from our i...
8,10171785,A,The Pont au Double is a cast iron bridge close...
9,10180824,A,It’s a small rectangular area- a small park. A...


---
## 4. Split Train / Test (50 / 50 par lieu)

In [10]:
from sklearn.model_selection import train_test_split

all_place_ids = place_documents['idplace'].unique()

train_ids, test_ids = train_test_split(
    all_place_ids,
    test_size=0.5,
    random_state=RANDOM_SEED
)

train_docs = place_documents[place_documents['idplace'].isin(train_ids)].reset_index(drop=True)
test_docs  = place_documents[place_documents['idplace'].isin(test_ids)].reset_index(drop=True)

print(f'Train : {len(train_docs):,} lieux')
print(f'Test  : {len(test_docs):,} lieux')
print('\nDistribution typeR — Train :', train_docs['typeR'].value_counts().to_dict())
print('Distribution typeR — Test  :', test_docs['typeR'].value_counts().to_dict())

Train : 917 lieux
Test  : 918 lieux

Distribution typeR — Train : {'AP': 494, 'R': 271, 'A': 118, 'H': 34}
Distribution typeR — Test  : {'AP': 495, 'R': 267, 'A': 134, 'H': 22}


---
## 5. Prétraitement NLP & Tokenisation

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def preprocess_text(text: str, stem: bool = True) -> List[str]:
    """
    Pipeline NLP minimal :
      1. Minuscules
      2. Suppression des caractères non-alphabétiques
      3. Tokenisation NLTK
      4. Suppression stopwords
      5. Stemming optionnel (Porter)
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Tokenisation des documents (corpus de test = index de recherche)
print('Tokenisation du corpus de test...')
test_docs['tokens'] = [preprocess_text(doc) for doc in tqdm(test_docs['document'])]

print('Tokenisation du corpus de train (requêtes)...')
train_docs['tokens'] = [preprocess_text(doc) for doc in tqdm(train_docs['document'])]

print(' Tokenisation terminée')

Tokenisation du corpus de test...


100%|██████████| 918/918 [00:08<00:00, 111.21it/s]


Tokenisation du corpus de train (requêtes)...


100%|██████████| 917/917 [00:07<00:00, 120.03it/s]

✅ Tokenisation terminée


---
## 6. Baseline BM25

> **Architecture** : le modèle est indexé sur les lieux du **set de test** et interrogé par les lieux du **set de train** (chaque lieu train = une requête).

In [ ]:
# ─── 6.1 Interface générique du modèle de recherche ──────────────────────────
class RetrievalModel:
    """
    Interface abstraite. Implémentez `build_index` et `rank` pour brancher
    n'importe quel modèle NLP (TF-IDF, SBERT, DPR, ColBERT...).
    """
    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        raise NotImplementedError

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        """
        Retourne un tableau de scores (un par document indexé).
        Score plus élevé = plus pertinent.
        """
        raise NotImplementedError


# ─── 6.2 Implémentation BM25 ──────────────────────────────────────────────────
class BM25Model(RetrievalModel):
    """
    Baseline BM25 (Okapi) via rank-bm25.
    k1=1.5, b=0.75 — paramètres standards TREC.
    """
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.bm25 = None

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        self.bm25 = BM25Okapi(corpus_tokens, k1=self.k1, b=self.b)
        print(f' Index BM25 construit — {len(corpus_tokens):,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        return self.bm25.get_scores(query_tokens)


# ─── 6.3 Construction de l'index ─────────────────────────────────────────────
bm25_model = BM25Model()
bm25_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

✅ Index BM25 construit — 918 documents


---
## 7. Fonctions d'évaluation

In [ ]:
# ─── 7.1 Helpers de correspondance de métadonnées ────────────────────────────

def match_level1(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    """Niveau 1 : même typeR (H, R, A, AP)."""
    return query_row['typeR'] == candidate_row['typeR']


def match_level2(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    """
    Niveau 2 : correspondance fine selon le type.
    Règle : au moins UNE catégorie commune dans les listes.
    """
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        """True si au moins un élément en commun."""
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):   # Attractions
        sub_cat   = lists_overlap(query_row.get('activiteSubCategorie', []),
                                  candidate_row.get('activiteSubCategorie', []))
        sub_type  = lists_overlap(query_row.get('activiteSubType', []),
                                  candidate_row.get('activiteSubType', []))
        return sub_cat or sub_type

    elif type_r == 'R':         # Restaurants
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine   = lists_overlap(query_row.get('cuisine', []),
                                  candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':         # Hôtels
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        if pd.isna(q_price) or pd.isna(c_price) or q_price == '' or c_price == '':
            return False
        return q_price == c_price

    return False  # Type inconnu


def has_valid_candidate(query_row: pd.Series,
                        test_df: pd.DataFrame,
                        match_fn: Callable) -> bool:
    """
    Vérifie qu'au moins un lieu du set de test peut correspondre à la requête.
    Si non → requête invalide (exclue de l'évaluation).
    """
    return any(match_fn(query_row, test_df.iloc[i]) for i in range(len(test_df)))


print('Fonctions de correspondance définies')

✅ Fonctions de correspondance définies


In [26]:
def match_level2_v2(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    """Version moins stricte pour les hôtels."""
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):
        sub_cat   = lists_overlap(query_row.get('activiteSubCategorie', []),
                                  candidate_row.get('activiteSubCategorie', []))
        sub_type  = lists_overlap(query_row.get('activiteSubType', []),
                                  candidate_row.get('activiteSubType', []))
        return sub_cat or sub_type

    elif type_r == 'R':
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine   = lists_overlap(query_row.get('cuisine', []),
                                  candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        
        # ✅ Si les deux ont un prix, ils doivent matcher
        if pd.notna(q_price) and pd.notna(c_price) and q_price != '' and c_price != '':
            return q_price == c_price
        
        # ✅ Si au moins un est vide, accepter tout hôtel
        # (fallback au niveau 1 : typeR='H')
        return True

    return False

# Tester
res_l2_v2 = evaluate_model(bm25_model, train_docs, test_docs, match_level2_v2, 'L2_v2')
print(f"L2 (assouplies) — Mean Error: {res_l2_v2['mean_error']:.2f} | Success@1: {res_l2_v2['success_at_1']:.1f}%")

Évaluation L2_v2: 100%|██████████| 917/917 [01:43<00:00,  8.82it/s]

L2 (assouplies) — Mean Error: 9.21 | Success@1: 70.7%


In [ ]:
def match_level2_ap_fallback(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    """
    Niveau 2 avec fallback pour AP (qui n'ont pas de sous-catégories).
    - A/AP : matchent sur sous-catégories OU fallback à L1
    - R : matchent sur type/cuisine
    - H : matchent sur priceRange avec fallback
    """
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):
        sub_cat = lists_overlap(query_row.get('activiteSubCategorie', []),
                                candidate_row.get('activiteSubCategorie', []))
        sub_type = lists_overlap(query_row.get('activiteSubType', []),
                                 candidate_row.get('activiteSubType', []))
        
        #  Match si catégories trouvées
        if sub_cat or sub_type:
            return True
        
        #  Fallback à L1 (juste typeR) — pour AP/A sans métadonnées
        return query_row['typeR'] == candidate_row['typeR']

    elif type_r == 'R':
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine = lists_overlap(query_row.get('cuisine', []),
                               candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        if pd.notna(q_price) and pd.notna(c_price) and q_price != '' and c_price != '':
            return q_price == c_price
        return True

    return False

# ─── Test ───────────────────────────────────────────────────────────────────
print('=' * 55)
print('ÉVALUATION NIVEAU 2 AP_FALLBACK — avec fallback AP→L1')
print('=' * 55)

res_l2_ap_fallback = evaluate_model(
    model      = bm25_model,
    train_df   = train_docs,
    test_df    = test_docs,
    match_fn   = match_level2_ap_fallback,
    level_name = 'L2_AP_fallback'
)

print(f"\n Résultats Niveau 2 AP_fallback :")
print(f"  Requêtes évaluées : {res_l2_ap_fallback['n_queries']:,}")
print(f"  Requêtes ignorées : {res_l2_ap_fallback['n_skipped']:,}")
print(f"  Ranking Error moyen   : {res_l2_ap_fallback['mean_error']:.2f}")
print(f"  Ranking Error médian  : {res_l2_ap_fallback['median_error']:.2f}")
print(f"  Success@1 (erreur=0)  : {res_l2_ap_fallback['success_at_1']:.1f}%")

ÉVALUATION NIVEAU 2 AP_FALLBACK — avec fallback AP→L1


Évaluation L2_AP_fallback: 100%|██████████| 917/917 [02:31<00:00,  6.07it/s]


✅ Résultats Niveau 2 AP_fallback :
  Requêtes évaluées : 914
  Requêtes ignorées : 3
  Ranking Error moyen   : 1.53
  Ranking Error médian  : 0.00
  Success@1 (erreur=0)  : 82.3%


In [ ]:
# ─── 7.2 Calcul du Ranking Error ─────────────────────────────────────────────

def compute_ranking_error(query_row: pd.Series,
                          scores: np.ndarray,
                          test_df: pd.DataFrame,
                          match_fn: Callable) -> Optional[int]:
    """
    Retourne le Ranking Error pour une requête :
      - 0 si le 1er résultat correspond
      - n-1 si la 1ère correspondance est en position n (0-indexée)
      - None si aucune correspondance n'existe dans le test set
    """
    ranked_indices = np.argsort(scores)[::-1]  # Tri décroissant

    for rank, idx in enumerate(ranked_indices):
        candidate = test_df.iloc[idx]
        if match_fn(query_row, candidate):
            return rank   # rank=0 → erreur=0 ; rank=n → erreur=n

    return None  # Pas de candidat valide


def evaluate_model(model: RetrievalModel,
                   train_df: pd.DataFrame,
                   test_df: pd.DataFrame,
                   match_fn: Callable,
                   level_name: str = 'L1') -> Dict:
    """
    Évalue un modèle de retrieval sur toutes les requêtes valides.

    Returns
    -------
    dict avec :
      - errors          : liste des ranking errors (hors None)
      - mean_error      : erreur moyenne
      - median_error    : erreur médiane
      - success_at_1    : % de requêtes avec erreur = 0
      - n_queries       : nombre de requêtes évaluées
      - n_skipped       : requêtes sans candidat valide
    """
    errors    = []
    n_skipped = 0

    for _, query_row in tqdm(train_df.iterrows(), total=len(train_df),
                             desc=f'Évaluation {level_name}'):

        # Vérification existence d'un candidat valide
        if not has_valid_candidate(query_row, test_df, match_fn):
            n_skipped += 1
            continue

        # Scores du modèle
        scores = model.rank(
            query_tokens=query_row['tokens'],
            query_text=query_row.get('document', '')
        )

        err = compute_ranking_error(query_row, scores, test_df, match_fn)
        if err is not None:
            errors.append(err)

    results = {
        'errors'       : errors,
        'mean_error'   : np.mean(errors)   if errors else float('nan'),
        'median_error' : np.median(errors) if errors else float('nan'),
        'success_at_1' : np.mean([e == 0 for e in errors]) * 100 if errors else 0.0,
        'n_queries'    : len(errors),
        'n_skipped'    : n_skipped,
    }
    return results


print(' Fonctions d\'évaluation définies')

✅ Fonctions d'évaluation définies


---
## 8. Évaluation BM25 — Niveau 1 & Niveau 2

In [ ]:
# ─── Niveau 1 : typeR ─────────────────────────────────────────────────────────
print('=' * 55)
print('ÉVALUATION NIVEAU 1 — typeR (H / R / A / AP)')
print('=' * 55)

results_l1 = evaluate_model(
    model      = bm25_model,
    train_df   = train_docs,
    test_df    = test_docs,
    match_fn   = match_level1,
    level_name = 'L1'
)

print(f"\n Résultats Niveau 1 :")
print(f"  Requêtes évaluées : {results_l1['n_queries']}")
print(f"  Requêtes ignorées : {results_l1['n_skipped']} (pas de candidat valide)")
print(f"  Ranking Error moyen   : {results_l1['mean_error']:.2f}")
print(f"  Ranking Error médian  : {results_l1['median_error']:.2f}")
print(f"  Success@1 (erreur=0)  : {results_l1['success_at_1']:.1f}%")

ÉVALUATION NIVEAU 1 — typeR (H / R / A / AP)


Évaluation L1: 100%|██████████| 917/917 [02:02<00:00,  7.51it/s]


📊 Résultats Niveau 1 :
  Requêtes évaluées : 917
  Requêtes ignorées : 0 (pas de candidat valide)
  Ranking Error moyen   : 0.64
  Ranking Error médian  : 0.00
  Success@1 (erreur=0)  : 87.8%


In [27]:
# ─── Évaluation avec la version v2 (assouplies pour hôtels) ──────────────────

print('=' * 55)
print('ÉVALUATION NIVEAU 2 v2 — Catégories fines (hôtels assouplies)')
print('=' * 55)

results_l2_v2 = evaluate_model(
    model      = bm25_model,
    train_df   = train_docs,
    test_df    = test_docs,
    match_fn   = match_level2_v2,  # ← Utiliser v2 ici
    level_name = 'L2_v2'
)

print(f"\nRésultats Niveau 2 v2 :")
print(f"  Requêtes évaluées : {results_l2_v2['n_queries']}")
print(f"  Requêtes ignorées : {results_l2_v2['n_skipped']}")
print(f"  Ranking Error moyen   : {results_l2_v2['mean_error']:.2f}")
print(f"  Ranking Error médian  : {results_l2_v2['median_error']:.2f}")
print(f"  Success@1 (erreur=0)  : {results_l2_v2['success_at_1']:.1f}%")

ÉVALUATION NIVEAU 2 v2 — Catégories fines (hôtels assouplies)


Évaluation L2_v2: 100%|██████████| 917/917 [01:44<00:00,  8.74it/s]


📊 Résultats Niveau 2 v2 :
  Requêtes évaluées : 410
  Requêtes ignorées : 507
  Ranking Error moyen   : 9.21
  Ranking Error médian  : 0.00
  Success@1 (erreur=0)  : 70.7%


In [29]:
# Analyser les requêtes ignorées par type
print("Distribution des ignores par typeR :")

for type_r in ['H', 'R', 'A', 'AP']:
    train_subset = train_docs[train_docs['typeR'] == type_r]
    
    # Compter combien n'ont pas de candidat valide
    n_ignored = sum([
        not has_valid_candidate(row, test_docs, match_level2_v2)
        for _, row in train_subset.iterrows()
    ])
    
    print(f"  {type_r:3s} — {n_ignored:3d}/{len(train_subset):3d} ignorées ({100*n_ignored/len(train_subset):.1f}%)")

Distribution des ignores par typeR :
  H   —   0/ 34 ignorées (0.0%)
  R   —   3/271 ignorées (1.1%)
  A   —  10/118 ignorées (8.5%)
  AP  — 494/494 ignorées (100.0%)


In [17]:
# ─── Tableau récapitulatif ────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Modèle'         : ['BM25', 'BM25'],
    'Niveau'         : ['L1 — typeR', 'L2 — catégories fines'],
    'Requêtes'       : [results_l1['n_queries'], results_l2['n_queries']],
    'Mean Error ↓'   : [round(results_l1['mean_error'], 3), round(results_l2['mean_error'], 3)],
    'Median Error ↓' : [round(results_l1['median_error'], 3), round(results_l2['median_error'], 3)],
    'Success@1 ↑ (%)'  : [round(results_l1['success_at_1'], 1), round(results_l2['success_at_1'], 1)],
})
print('\n', summary.to_string(index=False))


 Modèle                Niveau  Requêtes  Mean Error ↓  Median Error ↓  Success@1 ↑ (%)
  BM25            L1 — typeR       917         0.642             0.0             87.8
  BM25 L2 — catégories fines       408         9.392             0.0             70.6


---
## 9. Ajout d'un second modèle (plug-and-play)

Pour comparer un nouveau modèle à BM25, il suffit de :
1. Hériter de `RetrievalModel`
2. Implémenter `build_index` et `rank`
3. Appeler `evaluate_model(new_model, ...)`

### Exemple : TF-IDF cosinus (modèle alternatif démo)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class TFIDFCosineModel(RetrievalModel):
    """
    Modèle TF-IDF + similarité cosinus.
    Exemple de remplacement plug-and-play de BM25.
    """
    def __init__(self):
        self.vectorizer    = TfidfVectorizer(analyzer='word', stop_words='english')
        self.corpus_matrix = None

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        # TF-IDF travaille sur du texte brut
        docs = corpus_texts if corpus_texts else [' '.join(t) for t in corpus_tokens]
        self.corpus_matrix = self.vectorizer.fit_transform(docs)
        print(f' Index TF-IDF construit — {self.corpus_matrix.shape[0]:,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        query_str    = query_text if query_text else ' '.join(query_tokens)
        query_vector = self.vectorizer.transform([query_str])
        scores       = cosine_similarity(query_vector, self.corpus_matrix).flatten()
        return scores


# ─── Construction et évaluation ───────────────────────────────────────────────
tfidf_model = TFIDFCosineModel()
tfidf_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_tfidf_l1 = evaluate_model(tfidf_model, train_docs, test_docs, match_level1, 'TF-IDF L1')
res_tfidf_l2 = evaluate_model(tfidf_model, train_docs, test_docs, match_level2, 'TF-IDF L2')

print(f"\nTF-IDF L1 — Mean Error: {res_tfidf_l1['mean_error']:.2f} | Success@1: {res_tfidf_l1['success_at_1']:.1f}%")
print(f"TF-IDF L2 — Mean Error: {res_tfidf_l2['mean_error']:.2f} | Success@1: {res_tfidf_l2['success_at_1']:.1f}%")

✅ Index TF-IDF construit — 918 documents


Évaluation TF-IDF L2: 100%|██████████| 917/917 [00:23<00:00, 39.10it/s]


TF-IDF L1 — Mean Error: 0.79 | Success@1: 88.9%
TF-IDF L2 — Mean Error: 11.15 | Success@1: 70.8%


In [23]:
# ─── Tableau de comparaison final ────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Modèle': 'BM25',   'Niveau': 'L1', 'Mean Error': results_l1['mean_error'],    'Success@1 (%)': results_l1['success_at_1']},
    {'Modèle': 'BM25',   'Niveau': 'L2', 'Mean Error': results_l2['mean_error'],    'Success@1 (%)': results_l2['success_at_1']},
    {'Modèle': 'TF-IDF', 'Niveau': 'L1', 'Mean Error': res_tfidf_l1['mean_error'], 'Success@1 (%)': res_tfidf_l1['success_at_1']},
    {'Modèle': 'TF-IDF', 'Niveau': 'L2', 'Mean Error': res_tfidf_l2['mean_error'], 'Success@1 (%)': res_tfidf_l2['success_at_1']},
    {'Modèle': 'SBERT',  'Niveau': 'L1', 'Mean Error': res_sbert_l1['mean_error'],  'Success@1 (%)': res_sbert_l1['success_at_1']},
    {'Modèle': 'SBERT',  'Niveau': 'L2', 'Mean Error': res_sbert_l2['mean_error'],  'Success@1 (%)': res_sbert_l2['success_at_1']},
    {'Modèle': 'Hybrid', 'Niveau': 'L1', 'Mean Error': res_hybrid_l1['mean_error'], 'Success@1 (%)': res_hybrid_l1['success_at_1']},
    {'Modèle': 'Hybrid', 'Niveau': 'L2', 'Mean Error': res_hybrid_l2['mean_error'], 'Success@1 (%)': res_hybrid_l2['success_at_1']},
])

comparison['Mean Error'] = comparison['Mean Error'].round(3)
comparison['Success@1 (%)'] = comparison['Success@1 (%)'].round(1)
print('\n' + '═' * 70)
print(' COMPARAISON FINALE')
print('═' * 70)
print(comparison.to_string(index=False))


══════════════════════════════════════════════════════════════════════
 COMPARAISON FINALE
══════════════════════════════════════════════════════════════════════
Modèle Niveau  Mean Error  Success@1 (%)
  BM25     L1       0.642           87.8
  BM25     L2       9.392           70.6
TF-IDF     L1       0.785           88.9
TF-IDF     L2      11.154           70.8
 SBERT     L1       0.788           88.4
 SBERT     L2       9.735           68.6
Hybrid     L1       0.606           90.1
Hybrid     L2       9.319           70.1


---
## 9bis. SBERT (Sentence-BERT) — Embeddings sémantiques

In [20]:
# ─── Installation de sentence-transformers ──────────────────────────────────
%pip install sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class SBERTModel(RetrievalModel):
    """
    Sentence-BERT (Semantic embeddings).
    Utilise des embeddings pré-entraînés pour capturer la similarité sémantique.
    Modèle : all-MiniLM-L6-v2 (384 dim, léger et rapide).
    """
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.corpus_embeddings = None
        print(f"SBERT chargé : {model_name}")

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        # SBERT fonctionne sur du texte brut
        docs = corpus_texts if corpus_texts else [' '.join(t) for t in corpus_tokens]
        print(f"Encodage de {len(docs):,} documents SBERT...")
        self.corpus_embeddings = self.model.encode(docs, show_progress_bar=True, batch_size=32)
        print(f'Index SBERT construit — {len(docs):,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        query_str = query_text if query_text else ' '.join(query_tokens)
        query_emb = self.model.encode([query_str])
        scores = cosine_similarity(query_emb, self.corpus_embeddings).flatten()
        return scores


# ─── Construction et évaluation SBERT ────────────────────────────────────────
sbert_model = SBERTModel()
sbert_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_sbert_l1 = evaluate_model(sbert_model, train_docs, test_docs, match_level1, 'SBERT L1')
res_sbert_l2 = evaluate_model(sbert_model, train_docs, test_docs, match_level2, 'SBERT L2')

print(f"\nSBERT L1 — Mean Error: {res_sbert_l1['mean_error']:.2f} | Success@1: {res_sbert_l1['success_at_1']:.1f}%")
print(f"SBERT L2 — Mean Error: {res_sbert_l2['mean_error']:.2f} | Success@1: {res_sbert_l2['success_at_1']:.1f}%")

c:\Users\axell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 917.79it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT chargé : all-MiniLM-L6-v2
Encodage de 918 documents SBERT...


Batches: 100%|██████████| 29/29 [00:17<00:00,  1.68it/s]


✅ Index SBERT construit — 918 documents


Évaluation SBERT L2: 100%|██████████| 917/917 [00:33<00:00, 27.54it/s]


SBERT L1 — Mean Error: 0.79 | Success@1: 88.4%
SBERT L2 — Mean Error: 9.74 | Success@1: 68.6%


---
## 9ter. Modèle Hybrid — Fusion BM25 + SBERT

In [22]:
class HybridModel(RetrievalModel):
    """
    Fusion BM25 (lexical) + SBERT (sémantique).
    Combine : alpha * bm25_scores + (1-alpha) * sbert_scores (normalisés 0-1).
    """
    def __init__(self, bm25_model: BM25Model, sbert_model: SBERTModel, alpha: float = 0.5):
        self.bm25_model = bm25_model
        self.sbert_model = sbert_model
        self.alpha = alpha  # 0.5 = poids égal entre BM25 et SBERT

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        # Les deux modèles ont déjà leurs index, rien à faire ici
        print(f'Index Hybrid configuré (α={self.alpha} BM25 + {1-self.alpha} SBERT)')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        # Scores BM25
        bm25_scores = self.bm25_model.rank(query_tokens, query_text)
        # Scores SBERT
        sbert_scores = self.sbert_model.rank(query_tokens, query_text)
        
        # Normalisation des deux scores à [0, 1]
        bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
        sbert_norm = (sbert_scores - sbert_scores.min()) / (sbert_scores.max() - sbert_scores.min() + 1e-10)
        
        # Fusion
        hybrid_scores = self.alpha * bm25_norm + (1 - self.alpha) * sbert_norm
        return hybrid_scores


# ─── Construction et évaluation Hybrid ───────────────────────────────────────
hybrid_model = HybridModel(bm25_model, sbert_model, alpha=0.5)
hybrid_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_hybrid_l1 = evaluate_model(hybrid_model, train_docs, test_docs, match_level1, 'Hybrid L1')
res_hybrid_l2 = evaluate_model(hybrid_model, train_docs, test_docs, match_level2, 'Hybrid L2')

print(f"\nHybrid L1 — Mean Error: {res_hybrid_l1['mean_error']:.2f} | Success@1: {res_hybrid_l1['success_at_1']:.1f}%")
print(f"Hybrid L2 — Mean Error: {res_hybrid_l2['mean_error']:.2f} | Success@1: {res_hybrid_l2['success_at_1']:.1f}%")

Index Hybrid configuré (α=0.5 BM25 + 0.5 SBERT)


Évaluation Hybrid L2: 100%|██████████| 917/917 [01:51<00:00,  8.20it/s]


Hybrid L1 — Mean Error: 0.61 | Success@1: 90.1%
Hybrid L2 — Mean Error: 9.32 | Success@1: 70.1%


---
## 10. Analyse des erreurs (debug)

Inspecter les pires requêtes pour comprendre les échecs du modèle.

In [28]:
def inspect_worst_queries(model: RetrievalModel,
                          train_df: pd.DataFrame,
                          test_df: pd.DataFrame,
                          match_fn: Callable,
                          top_k: int = 5) -> pd.DataFrame:
    
    records = []
    for _, query_row in train_df.iterrows():
        if not has_valid_candidate(query_row, test_df, match_fn):
            continue
        scores = model.rank(query_row['tokens'], query_row.get('document', ''))
        err = compute_ranking_error(query_row, scores, test_df, match_fn)
        if err is not None:
            records.append({
                'idplace'       : query_row['idplace'],
                'typeR'         : query_row.get('typeR', ''),
                'ranking_error' : err,
                'doc_snippet'   : str(query_row.get('document', ''))[:120]
            })

    df = pd.DataFrame(records).sort_values('ranking_error', ascending=False).head(top_k)
    return df


worst = inspect_worst_queries(bm25_model, train_docs, test_docs, match_level1, top_k=5)
print('Top 5 requêtes avec la plus grande erreur (BM25, L1) :')
worst

Top 5 requêtes avec la plus grande erreur (BM25, L1) :


,idplace,typeR,ranking_error,doc_snippet
756,3889679,A,74,"Tasty down to earth tasty local Parisian food,..."
24,10432669,A,53,Not sure what anyone could find to complain ab...
776,4730092,A,43,We very much enjoyed our short but sweet trip ...
686,2254170,A,42,This place was recommended to me by a friend a...
10,10240156,A,34,"Lively crowd and good cocktails, worth a visit..."


---
## 11. Récapitulatif de l'architecture

```
📁 Pipeline
├── Data Layer
│   ├── reviews83325.csv  ──► filtre langue EN
│   └── Tripadvisor.csv   ──► jointure via idplace/id
│
├── Preprocessing
│   ├── strategy_cap()          ← normalisation avis (CAP)
│   └── strategy_tfidf_keywords() ← normalisation avis (TF-IDF)
│
├── Split  →  50% Train (requêtes) / 50% Test (index)
│
├── RetrievalModel  [interface]
│   ├── BM25Model           ← baseline
│   ├── TFIDFCosineModel    ← alternatif démo
│   └── YourModel(...)      ← branchez votre SBERT / DPR ici
│
└── Evaluation
    ├── match_level1()    → typeR
    ├── match_level2()    → sous-catégories fines
    └── evaluate_model()  → mean/median error, Success@1
```

### Pour brancher SBERT (Sentence-BERT) :
```python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class SBERTModel(RetrievalModel):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.corpus_embeddings = None

    def build_index(self, corpus_tokens, corpus_texts=None):
        texts = corpus_texts or [' '.join(t) for t in corpus_tokens]
        self.corpus_embeddings = self.model.encode(texts, show_progress_bar=True)

    def rank(self, query_tokens, query_text=None):
        q = query_text or ' '.join(query_tokens)
        q_emb = self.model.encode([q])
        return cosine_similarity(q_emb, self.corpus_embeddings).flatten()
```
Puis simplement :
```python
sbert = SBERTModel()
sbert.build_index(corpus_tokens=test_docs['tokens'].tolist(),
                  corpus_texts=test_docs['document'].tolist())
evaluate_model(sbert, train_docs, test_docs, match_level1)
```